# Session 14 — reproduce Linear → MoE

Select **Runtime → Change runtime type → GPU**, then **Run all**. This notebook clones only the small-model folder, creates an isolated Python 3.12 environment, checks the data/checkpoints, trains the baseline and both continuation arms, runs the 22 interventions and six balancing arms, and optionally reruns the two-seed cost study. No 20M model or Git LFS download is needed.

The local Windows/CUDA workflow has been tested. Hosted Colab execution has not yet been independently verified; runtime availability and hardware vary. The notebook records actual GPU/software details and reports reference drift instead of hiding it. No training results are prefilled in this notebook.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, json, shutil, uuid
RUN_FRESH_TRAINING = True
RUN_TWO_SEED_STUDY = True
SAVE_TO_DRIVE = False  # Set True to keep outputs in your Google Drive; requires normal Drive authorization.
ALLOW_REFERENCE_DRIFT = True  # Still checks correctness and loss reduction; records any numerical mismatch.
# For local validation only: set both overrides to existing paths. Colab users leave them None.
LOCAL_PACKAGE_OVERRIDE = None
LOCAL_PYTHON_OVERRIDE = None
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
WORK = (Path('/content') if Path('/content').exists() else Path.cwd()) / ('slm_' + SESSION_ID)
WORK.mkdir(parents=True, exist_ok=False)
RUN_ROOT = WORK/'outputs'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/ERA_Week14_Small_MoE')/SESSION_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Output directory:', RUN_ROOT)


## 1. Obtain the standalone package

In [ ]:
if LOCAL_PACKAGE_OVERRIDE:
    PACKAGE = Path(LOCAL_PACKAGE_OVERRIDE).resolve()
else:
    repo = WORK/'ERAv5'
    git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    'https://github.com/udisinghania/ERAv5.git',str(repo)],check=True,env=git_env)
    subprocess.run(['git','-C',str(repo),'sparse-checkout','set','Week 14/Small_Linear_MoE'],check=True,env=git_env)
    PACKAGE = repo/'Week 14/Small_Linear_MoE'
assert (PACKAGE/'run.py').is_file()
assert (PACKAGE/'src/extended_study.py').is_file(), 'This notebook needs the README-completion update; push the updated package first.'
print('Package:', PACKAGE)


## 2. Create the pinned environment

The training process uses its own environment rather than changing the notebook kernel's NumPy/PyTorch. If the kernel is not Python 3.12, uv obtains a compatible interpreter. Dependency installation and cloning require internet access.


In [ ]:
if LOCAL_PYTHON_OVERRIDE:
    PYTHON = str(Path(LOCAL_PYTHON_OVERRIDE).resolve())
else:
    venv = WORK/'venv'
    if sys.version_info[:2] == (3,12):
        subprocess.run([sys.executable,'-m','venv',str(venv)],check=True)
    else:
        subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
        subprocess.run([sys.executable,'-m','uv','venv','--seed','--python','3.12',str(venv)],check=True)
    PYTHON = str(venv/('Scripts/python.exe' if os.name=='nt' else 'bin/python'))
    subprocess.run([PYTHON,'-m','pip','install','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
    subprocess.run([PYTHON,'-m','pip','install','numpy==1.26.4'],check=True)
subprocess.run([PYTHON,'-c',"import torch,numpy,sys; assert sys.version_info[:2]==(3,12); assert torch.__version__.split('+')[0]=='2.5.1'; assert numpy.__version__=='1.26.4'; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(sys.version); print(torch.__version__, numpy.__version__); print(torch.cuda.get_device_name(0))"],check=True)


## 3. Verify every file and evaluate the shipped checkpoints

In [ ]:
def run_command(command, output, allow_drift=False):
    args = [PYTHON,str(PACKAGE/'run.py'),command,'--output',str(output)]
    if allow_drift: args += ['--allow-reference-drift']
    subprocess.run(args,check=True)
subprocess.run([PYTHON,str(PACKAGE/'run.py'),'verify'],check=True)
run_command('evaluate',RUN_ROOT/'shipped_evaluation',ALLOW_REFERENCE_DRIFT)
print(json.loads((RUN_ROOT/'shipped_evaluation/evaluation.json').read_text()))


## 4. Train from scratch and rerun the deep dive

This is fresh training, not inference on the provided checkpoint. It produces new checkpoints and all diagnostic outputs in a new folder. Exact-reference comparisons remain visible. `REFERENCE_DRIFT` means the run finished but did not match the recorded losses within 1e-4; it must not be reported as exact reproduction.


In [ ]:
if RUN_FRESH_TRAINING:
    run_command('reproduce',RUN_ROOT/'fresh_reproduction',ALLOW_REFERENCE_DRIFT)
    reproduction = json.loads((RUN_ROOT/'fresh_reproduction/REPRODUCTION.json').read_text())
    print('Fresh reproduction status:',reproduction['status'])
    for arm,score in reproduction['evaluation']['scores'].items():print(arm,score['loss'])
    print('Deep-dive comparison:',reproduction['deep_dive'])
else:
    print('Fresh training was disabled; the previous cell evaluates shipped weights only.')


## 5. Rerun the two-seed cost study

This adds two fresh baseline runs and two dense/MoE continuation pairs. Data are fixed; initialization and batch sampling vary by seed. Measured throughput and memory are specific to the GPU/runtime assigned to you, and will differ from the laptop table. These two seeds are a small repeatability check, not a confidence interval.


In [ ]:
if RUN_TWO_SEED_STUDY:
    run_command('study',RUN_ROOT/'two_seed_study',ALLOW_REFERENCE_DRIFT)
    for row in json.loads((RUN_ROOT/'two_seed_study/results.json').read_text())['stages']:
        print(row['seed'],row['arm'],'loss=',round(row['history'][-1]['loss'],6),
              'targets/s=',round(row['training_targets_per_second']),
              'peak MiB=',round(row['peak_training_allocated_bytes']/2**20,2))
    print('Study status:',json.loads((RUN_ROOT/'two_seed_study/verification.json').read_text())['status'])
else:
    print('Two-seed study disabled.')


## 6. Preserve outputs

The archive contains the actual logs, metrics and newly produced checkpoints. Download it from Colab's Files pane, or use the optional download cell. If Drive was enabled, the original outputs also remain in the chosen Drive folder. Without Drive/download, a runtime reset can remove local outputs.


In [ ]:
archive = shutil.make_archive(str(WORK/'small_linear_moe_outputs'),'zip',RUN_ROOT)
print('Archive:',archive)
print('Results:',RUN_ROOT)
# Optional in Colab:
# from google.colab import files
# files.download(archive)
